In [0]:
%run ../gold/gold

In [0]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
from pyspark.sql  import functions as F
dbutils.widgets.text("catalog", "workspace")

catalog = dbutils.widgets.get("catalog")
silver_orders = f"{catalog}.silver.orders"



In [0]:
if not spark.catalog.tableExists(silver_orders):
    raise Exception(f"{silver_orders} does not exist. Run the silver notebook.")


silver_df = spark.table(silver_orders)
if silver_df.isEmpty():
    dbutils.notebook.exit("silver.orders is empty. nothing to aggregate.")

print(f"silver_orders: {silver_df.count()}")



In [0]:
bronze_payments = (
    spark.table(f"{catalog}.bronze.raw_files")
    .select("order_id", "payment_method")
    .dropDuplicates(["order_id"])
)
gold_input = silver_df.join(bronze_payments, "order_id", "left")

gold_dfs = build_gold(gold_input)

for name in GOLD_TABLES:
    target = f"{catalog}.gold.{name}"
    (gold_dfs[name].write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target))
    print(f"written {target}: {spark.table(target).count()} rows")

In [0]:
expected = total_net_revenue(silver_df)

print(f"silver net revenue (revenue orders only): {expected}")
problems = []
for name in RECONCILE_TABLES:
   actual = spark.table(f"{catalog}.gold.{name}").agg(F.sum("net_revenue")).collect()[0][0]
   status = "OK" if actual == expected else "MISMATCH"
   print(f"{name:26s} {actual} {status}")
   if actual != expected:
       problems.append(name)

assert not problems, f"Gold does not reconcile with silver for {problems}"


In [0]:
display(spark.table(f"{catalog}.gold.monthly_category_sales"))

In [0]:
display(spark.table(f"{catalog}.gold.product_performance").limit(5))

In [0]:
display(spark.table(f"{catalog}.gold.customer_summary").limit(10))

In [0]:
display(spark.table(f"{catalog}.gold.payment_summary"))